# Phase 3 — Text Preprocessing


### Goal

Build one reproducible processed dataset that keeps several text representations for later experiments while avoiding model-specific preprocessing.


## Table of Content
- 

## 1. Imports and notebook configuration


In [1]:
from pathlib import Path
import sys

import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 140)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

print("Pandas:", pd.__version__)

Pandas: 3.0.5


In [2]:
def locate_project_root(start: Path | None = None) -> Path:
    """Locate the repository root by finding the 1_data_acquisition folder."""
    start_path = (start or Path.cwd()).resolve()

    for candidate in [start_path, *start_path.parents]:
        if (candidate / "1_data_acquisition").exists():
            return candidate

    raise FileNotFoundError(
        "Could not locate the project root. Run this notebook inside the repository."
    )


PROJECT_ROOT = locate_project_root()
PHASE3_DIR = PROJECT_ROOT / "3_text_preprocessing"

if str(PHASE3_DIR) not in sys.path:
   sys.path.insert(0, str(PHASE3_DIR))

from src.preprocessing_utils import (
    PRIMARY_FILENAME,
    GENERATED_TEXT_COLUMNS,
    resolve_historical_dataset,
    default_processed_output_path,
    resolve_schema,
    validate_core_schema,
    audit_coin_type_values,
    explicit_bitcoin_labels,
    preprocess_csv_to_parquet,
    validation_summary,
    financial_text_smoke_test,
)

print("Project root:", PROJECT_ROOT)
print("Phase 3 directory:", PHASE3_DIR)

Project root: C:\Users\sepehr\PycharmProjects\FinancialNLP
Phase 3 directory: C:\Users\sepehr\PycharmProjects\FinancialNLP\3_text_preprocessing


## 2. Input and output configuration

### Question / Goal

Which file is Phase 3 allowed to read, and where should the generated dataset be written?

### Decision

- Read the pinned CryptoVision V1 raw CSV from Phase 1.
- Never overwrite that file.
- Write a new Parquet file inside Phase 3's generated-data directory.

In [3]:
CHUNK_SIZE = None

V1_PATH = resolve_historical_dataset(PROJECT_ROOT, PRIMARY_FILENAME)
OUTPUT_PATH = default_processed_output_path(PROJECT_ROOT)

print("Raw input:", V1_PATH)
print("Processed output:", OUTPUT_PATH)
print("Raw file exists:", V1_PATH.exists())
print("Chunked mode:", CHUNK_SIZE is not None)

if V1_PATH.resolve() == OUTPUT_PATH.resolve():
    raise RuntimeError("Raw and processed paths must never be the same file.")


Raw input: C:\Users\sepehr\PycharmProjects\FinancialNLP\1_data_acquisition\historical\row\cryptovision_v1.csv
Processed output: C:\Users\sepehr\PycharmProjects\FinancialNLP\3_text_preprocessing\data\processed\cryptovision_v1_preprocessed.parquet
Raw file exists: True
Chunked mode: False


## 3. Validate the real CryptoVision V1 schema

### Question / Goal

What columns are actually present in the raw file, and do the fields required by Phase 3 exist?

### Context from Phase 2

The real V1 file uses names such as `Full Text`, `Date Time`, and `Coin Type`. Earlier exploratory assumptions may use underscore variants, so Phase 3 validates the current file instead of silently trusting an old schema assumption.

In [4]:
raw_columns = pd.read_csv(V1_PATH, nrows=0).columns.tolist()
schema = validate_core_schema(raw_columns)
resolved_schema = resolve_schema(raw_columns)

print(f"Observed columns: {len(raw_columns)}")
for index, column in enumerate(raw_columns, start=1):
    print(f"{index:>2}. {column}")

pd.DataFrame(
    [
        {
            "semantic_field": semantic,
            "resolved_column": column,
            "required_in_phase_3": semantic in schema,
        }
        for semantic, column in resolved_schema.items()
    ]
)

Observed columns: 17
 1. URL
 2. Title
 3. Description
 4. Full Text
 5. Date Time
 6. Coin Type
 7. Filtered_Text
 8. sentiment_label
 9. sentiment_score
10. Open
11. High
12. Low
13. Close
14. Volume
15. Movement_OpenClose_%
16. Movement_HighLow_%
17. Market_Move


,semantic_field,resolved_column,required_in_phase_3
0,url,URL,True
1,title,Title,True
2,description,Description,True
3,full_text,Full Text,True
4,date_time,Date Time,True
5,coin_type,Coin Type,True
6,filtered_text,Filtered_Text,True
7,sentiment_label,sentiment_label,False
8,sentiment_score,sentiment_score,False


### Interpretation

Phase 3 uses semantic roles internally but preserves the original dataset column names in the output. This prevents accidental raw-schema rewriting while keeping the code resilient and readable.

### Decision

If a core field is missing, stop and investigate the acquisition/schema version instead of guessing.

## 4. Confirm the raw row count

### Question / Goal

How many rows are present before any Phase 3 filtering?

### Reason

A before/after row count is necessary to explain every reduction caused by Bitcoin selection, missing-text handling, and deduplication.

In [5]:
url_column = schema["url"]

if CHUNK_SIZE is None:
    raw_row_count = len(
        pd.read_csv(
            V1_PATH,
            usecols=[url_column],
            low_memory=False,
        )
    )
else:
    raw_row_count = sum(
        len(chunk)
        for chunk in pd.read_csv(
            V1_PATH,
            usecols=[url_column],
            chunksize=CHUNK_SIZE,
        )
    )

print(f"Raw V1 rows: {raw_row_count:,}")


Raw V1 rows: 188,430


## 5. Audit actual `Coin Type` values before selecting Bitcoin

### Question / Goal

How does this dataset actually encode cryptocurrency categories?

### Why this matters

A substring filter such as `contains("bitcoin")` can incorrectly include `Bitcoin Cash` or `Bitcoin SV`. We first inspect the real values, then use only labels with an explicit Bitcoin/BTC component.

In [6]:
coin_column = schema["coin_type"]
coin_counts = audit_coin_type_values(
    V1_PATH,
    coin_column=coin_column,
    chunksize=CHUNK_SIZE,
)

coin_counts.head(50)

,Coin Type,count,is_explicit_bitcoin_label
0,Bitcoin,90346,True
1,Ethereum,38175,False
2,Solana,17348,False
3,Ripple,6545,False
4,Polygon,5179,False
5,Dogecoin,4913,False
6,Cardano,3936,False
7,Terra,3606,False
8,Avalanche,3176,False
9,Litecoin,2840,False


In [7]:
bitcoin_labels = explicit_bitcoin_labels(coin_counts, coin_column)

bitcoin_label_table = coin_counts[
    coin_counts["is_explicit_bitcoin_label"].fillna(False)
].copy()

print("Accepted full Coin Type labels:")
for value in bitcoin_labels:
    print("-", value)

bitcoin_label_table

Accepted full Coin Type labels:
- Bitcoin


,Coin Type,count,is_explicit_bitcoin_label
0,Bitcoin,90346,True


### Interpretation

The accepted values above come from the **observed dataset**, not from a hard-coded assumption about all possible crypto labels.

### Decision

Only these observed explicit Bitcoin/BTC labels will be passed to the preprocessing pipeline. If this table is empty or surprising, stop here and inspect the dataset instead of running the transformation.

In [8]:
if not bitcoin_labels:
    raise ValueError(
        "No explicit Bitcoin/BTC Coin Type labels were found. Review the values above."
    )

## 6. Define the canonical cleaning boundary

### Question / Goal

What should be cleaned now, and what should deliberately remain untouched until model-specific experimentation?

### Phase 3 cleaning

We allow only conservative, model-agnostic operations:

- HTML entity decoding and HTML tag/noise removal,
- Unicode NFKC normalization,
- zero-width character removal,
- embedded URL replacement with `<URL>`,
- whitespace normalization.

### Deliberately not applied

- stemming,
- stopword removal,
- number removal,
- punctuation removal,
- aggressive lowercasing,
- TF-IDF/tokenization/embeddings.

Numbers, percentages, currency symbols, punctuation, case, tickers, and financial terms are potentially informative and should survive the canonical cleaning step.

In [9]:
smoke_test = financial_text_smoke_test()
smoke_test

,before,after
0,"BTC jumps 5.2% to $67,500 after SEC ETF update.","BTC jumps 5.2% to $67,500 after SEC ETF update."
1,<p>ETH/BTC ratio rises; read more at https://example.com/a?utm_source=x</p>,ETH/BTC ratio rises; read more at <URL>
2,Fed says inflation is 3.1% — crypto markets remain volatile.,Fed says inflation is 3.1% — crypto markets remain volatile.


In [10]:
first_cleaned = smoke_test.loc[0, "after"]
assert "BTC" in first_cleaned
assert "5.2%" in first_cleaned
assert "$67,500" in first_cleaned
assert "SEC" in first_cleaned
assert "ETF" in first_cleaned

print("Financial-information preservation smoke test passed.")

Financial-information preservation smoke test passed.


### Decision

`Title` and `Description` receive this conservative cleaning. `Filtered_Text` is preserved **unchanged** as a publisher-provided candidate representation so it can be compared later with our own preprocessing.

## 7. Build the processed dataset

### Question / Goal

Can we create one dataset that:

1. selects audited Bitcoin rows,
2. preserves raw columns and metadata,
3. creates reusable clean text representations,
4. handles missing text conservatively,
5. removes repeated article identities,
6. writes a reproducible Parquet output?

### Duplicate policy

- Primary identity: normalized article URL.
- For duplicate URLs, keep the row with the richest available text; source order breaks ties.
- If URL is missing, use exact normalized `Title + Description` as a fallback.
- Do **not** remove same-title stories that have different URLs; they may be legitimate syndication or follow-up reporting.

In [11]:
processed_df, processing_summary = preprocess_csv_to_parquet(
    V1_PATH,
    OUTPUT_PATH,
    bitcoin_labels=bitcoin_labels,
    chunksize=CHUNK_SIZE,
)

pd.DataFrame(
    [{"metric": key, "value": value} for key, value in processing_summary.items()]
)

Language detection: 5,000/90,346 rows
Language detection: 10,000/90,346 rows
Language detection: 15,000/90,346 rows
Language detection: 20,000/90,346 rows
Language detection: 25,000/90,346 rows
Language detection: 30,000/90,346 rows
Language detection: 35,000/90,346 rows
Language detection: 40,000/90,346 rows
Language detection: 45,000/90,346 rows
Language detection: 50,000/90,346 rows
Language detection: 55,000/90,346 rows
Language detection: 60,000/90,346 rows
Language detection: 65,000/90,346 rows
Language detection: 70,000/90,346 rows
Language detection: 75,000/90,346 rows
Language detection: 80,000/90,346 rows
Language detection: 85,000/90,346 rows
Language detection: 90,000/90,346 rows
Language detection: 90,346/90,346 rows


,metric,value
0,raw_rows,188430
1,bitcoin_rows_selected,90346
2,non_bitcoin_rows_removed,98084
3,missing_all_candidate_text_rows_removed,0
4,language_rows_audited,90346
5,language_distribution_before_filter,"{'en': 89461, 'es': 345, 'de': 231, 'ru': 198, 'unknown': 29, 'nl': 24, 'bg': 15, 'fr': 13, 'ca': 5, 'da': 5, 'no': 4, 'af': 4, 'sv': 4,..."
6,english_rows_selected,89461
7,non_english_or_unknown_rows_removed,885
8,url_duplicate_rows_before_dedup,525
9,url_duplicate_pct_before_dedup,0.5868


In [12]:
language_distribution = pd.DataFrame(
    [
        {
            "language": language,
            "rows": rows,
        }
        for language, rows in processing_summary[
            "language_distribution_before_filter"
        ].items()
    ]
)

language_distribution["percentage"] = (
    language_distribution["rows"]
    / processing_summary["language_rows_audited"]
    * 100
)

language_distribution = language_distribution.sort_values(
    "rows",
    ascending=False,
    kind="stable",
).reset_index(drop=True)

language_distribution

,language,rows,percentage
0,en,89461,99.0204
1,es,345,0.3819
2,de,231,0.2557
3,ru,198,0.2192
4,unknown,29,0.0321
5,nl,24,0.0266
6,bg,15,0.0166
7,fr,13,0.0144
8,ca,5,0.0055
9,da,5,0.0055


In [14]:
language_filter_summary = pd.DataFrame(
    [
        {
            "metric": "rows_checked_for_language",
            "value": processing_summary["language_rows_audited"],
        },
        {
            "metric": "english_rows_kept",
            "value": processing_summary["english_rows_selected"],
        },
        {
            "metric": "non_english_or_unknown_rows_removed",
            "value": processing_summary["non_english_or_unknown_rows_removed"],
        },
    ]
)

language_filter_summary

,metric,value
0,rows_checked_for_language,90346
1,english_rows_kept,89461
2,non_english_or_unknown_rows_removed,885


### Interpretation

The summary above is the row-accounting record for Phase 3. It distinguishes:

- non-Bitcoin rows removed by scope,
- rows with no usable text candidate,
- URL duplicate candidates,
- URL + Title duplicate candidates,
- rows actually removed by the deduplication policy.

### Decision

The raw CSV remains untouched. The Parquet file is now the canonical handoff dataset for later phases.

## 8. Validate the final processed dataset

### Question / Goal

Did preprocessing produce a structurally valid dataset without leaving duplicate URL identities or rows with no usable text candidate?

In [15]:
checks = validation_summary(
    processed_df,
    url_column=schema["url"],
    filtered_text_column=schema["filtered_text"],
)

checks

,check,value
0,rows,88936
1,duplicate_normalized_url_rows,0
2,empty_title_clean,85
3,empty_description_clean,11015
4,empty_full_text_clean,0
5,empty_text_title,85
6,empty_text_title_description,76
7,empty_publisher_filtered_text,1
8,empty_all_text_candidates,0


In [17]:
check_map = dict(zip(checks["check"], checks["value"], strict=False))

assert OUTPUT_PATH.exists(), "Processed Parquet file was not created."
assert len(processed_df) == processing_summary["output_rows"]
assert check_map["duplicate_normalized_url_rows"] == 0
assert check_map["empty_all_text_candidates"] == 0
assert all(column in processed_df.columns for column in GENERATED_TEXT_COLUMNS)


print("Core Phase 3 validation checks passed.")

Core Phase 3 validation checks passed.


### Interpretation

Some individual representations can legitimately be empty. For example, a record with a missing title can have an empty `text_title` while still having a valid description or publisher `Filtered_Text`.

The important invariant is that the final row has at least one usable candidate representation and no repeated normalized URL identity.

## 9. Representative before/after examples

### Question / Goal

Does the transformation look conservative on real news records?

### Reason

Aggregate counts cannot reveal every cleaning mistake. A small deterministic sample lets us visually inspect case, punctuation, numbers, currency symbols, finance terminology, HTML cleanup, and URL replacement.

In [18]:
example_columns = [
    schema["title"],
    "title_clean",
    schema["description"],
    "description_clean",
    "text_title_description",
    schema["filtered_text"],
    "full_text_clean"
]

sample_size = min(10, len(processed_df))
processed_df[example_columns].sample(sample_size, random_state=42)

,Title,title_clean,Description,description_clean,text_title_description,Filtered_Text,full_text_clean
6342,"Long-Term Bitcoin Price Indicator Turns Bearish, Suggesting Bottom May Be In","Long-Term Bitcoin Price Indicator Turns Bearish, Suggesting Bottom May Be In","Bitcoin likely bottomed out in December and could begin a new bull run this year, as a lagging indicator has turned bearish for the firs...","Bitcoin likely bottomed out in December and could begin a new bull run this year, as a lagging indicator has turned bearish for the firs...","Long-Term Bitcoin Price Indicator Turns Bearish, Suggesting Bottom May Be In Bitcoin likely bottomed out in December and could begin a n...",likely bottomed december could begin new bull run year lagging indicator turned bearish first time four years,"Bitcoin likely bottomed out in December and could begin a new bull run this year, as a lagging indicator has turned bearish for the firs..."
7656,"BTC/USD Price Prediction — $6,000 Was Broken but What About $400,000 in the Long Term?","BTC/USD Price Prediction — $6,000 Was Broken but What About $400,000 in the Long Term?","A breakout of $6,000 and a $100 billion market cap – what to expect next from Bitcoin? Our detailed price analysis gives you the answer","A breakout of $6,000 and a $100 billion market cap – what to expect next from Bitcoin? Our detailed price analysis gives you the answer","BTC/USD Price Prediction — $6,000 Was Broken but What About $400,000 in the Long Term? A breakout of $6,000 and a $100 billion market ca...",breakout billion market cap expect next detailed price analysis gives answer,"A breakout of $6,000 and a $100 billion market cap – what to expect next from Bitcoin? Our detailed price analysis gives you the answer"
16703,"There Are Now Nearly 100,000 Bitcoin Millionaires","There Are Now Nearly 100,000 Bitcoin Millionaires","Bitcoin is one of the most profitable financial instruments of all time, but tracking how many people got rich from it is challenging. |...","Bitcoin is one of the most profitable financial instruments of all time, but tracking how many people got rich from it is challenging. |...","There Are Now Nearly 100,000 Bitcoin Millionaires Bitcoin is one of the most profitable financial instruments of all time, but tracking ...",short lifespan thebitcoin pricehas gone less cent high nearly unsurprisingly made people rich along way n't clear-cut exactlyhow manyluc...,"In its short lifespan, theBitcoin pricehas gone from less than a cent to highs of nearly $59,000. Unsurprisingly, it’s made some people ..."
39209,Don’t mention ‘K’ country: Bitcoin Magazine's YouTube restored after ban,Don’t mention ‘K’ country: Bitcoin Magazine's YouTube restored after ban,"After Bitcoin Magazine’s YouTube channel was temporarily shut down without prior warning, the livestream’s hosts suggested that mentions...","After Bitcoin Magazine’s YouTube channel was temporarily shut down without prior warning, the livestream’s hosts suggested that mentions...",Don’t mention ‘K’ country: Bitcoin Magazine's YouTube restored after ban After Bitcoin Magazine’s YouTube channel was temporarily shut d...,magazine youtube channel temporarily shut without prior warning livestream hosts suggested mentions kazakhstan may flagged platform algo...,"After Bitcoin Magazine’s YouTube channel was temporarily shut down without prior warning, the livestream’s hosts suggested that mentions..."
58937,"Nearly All Short-Term Bitcoin Owners Are Underwater, Making Rallies Harder","Nearly All Short-Term Bitcoin Owners Are Underwater, Making Rallies Harder","""Sharp upticks in short-term holder supply in loss tend to follow 'top heavy markets' such as May 2021, Dec 2021, and again this week,"" ...","""Sharp upticks in short-term holder supply in loss tend to follow 'top heavy markets' such as May 2021, Dec 2021, and again this week,"" ...","Nearly All Short-Term Bitcoin Owners Are Underwater, Making Rallies Harder ""Sharp upticks in short-term holder supp

### Decision

If visual inspection reveals systematic content loss, update the **single conservative cleaning function** in `src/preprocessing_utils.py` and rerun the notebook. Do not add ad hoc cleaning cells that make the pipeline difficult to reproduce.

## 10. Inspect the final schema and output file

### Question / Goal

What exactly will Phase 4 and Phase 5 receive?

In [20]:
print("Processed shape:", processed_df.shape)
print("Output path:", OUTPUT_PATH)
print("Output size (MB):", OUTPUT_PATH.stat().st_size / (1024**2))

pd.DataFrame(
    {
        "column": processed_df.columns,
        "dtype": [str(dtype) for dtype in processed_df.dtypes],
    }
)

Processed shape: (88936, 23)
Output path: C:\Users\sepehr\PycharmProjects\FinancialNLP\3_text_preprocessing\data\processed\cryptovision_v1_preprocessed.parquet
Output size (MB): 374.8876304626465


,column,dtype
0,source_row_id,int64
1,URL,str
2,Title,str
3,Description,str
4,Full Text,str
5,Date Time,str
6,Coin Type,str
7,Filtered_Text,str
8,sentiment_label,str
9,sentiment_score,float64


## 11. Final representation contract

The processed dataset keeps the original V1 columns and adds:

```text
source_row_id

title_clean
description_clean
text_title
text_title_description
full_text_clean
```

`Filtered_Text` remains in its original publisher-provided form and is a separate candidate representation.

### Recommended experiment candidates for Phase 4

```text
text_title
text_title_description
Filtered_Text
full_text_clean
```

Phase 4 should choose a representation **explicitly per experiment** and then perform model-specific feature extraction.

## 12. Leakage boundary and Phase 5 handoff

### Important boundary

Retaining metadata and market columns in the processed dataset does **not** make them valid features for the initial text-only model.

Publisher-provided fields such as sentiment annotations and OHLCV/market-move columns remain available for:

- auditing,
- comparison,
- later labeling research.

They must not be inserted into the initial text-only feature matrix.

### Phase 5 handoff

Phase 5 (`5_market_alignment_and_labeling`) should use the retained publication time and relevant market metadata/data sources to construct this project's own event-aligned targets, including candidate horizons such as:

```text
1h
4h
24h
```

No market-impact label is constructed in this notebook.

# Phase 3 complete

At this point the project has a clean, reproducible, deduplicated Bitcoin-focused dataset without committing to any one NLP model.

The next phase can compare multiple feature representations from the **same processed dataset**, while the market-labeling phase can independently build event-aligned outcomes from retained metadata.